In [1]:
!curl -o google-chrome-stable_current_amd64.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!sudo apt install ./google-chrome-stable_current_amd64.deb -y

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  119M  100  119M    0     0   333M      0 --:--:-- --:--:-- --:--:--  334M
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of './google-chrome-stable_current_amd64.deb'
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 12 newly installed, 0 to remove and 2 not upgraded.
Need to get 11.2 MB/136 MB of arc

In [4]:
!pip3 install selenium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 9.4 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


- https://habr.com/ru/companies/otus/articles/596071/

In [5]:
import selenium
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

- https://habr.com/ru/articles/513966/
- https://docs.python.org/3/howto/logging.html#configuring-logging

In [7]:
import logging
from logging.config import fileConfig

import os

In [56]:
os.makedirs('logs', exist_ok=True)

fileConfig('./logs/logging_config.ini')

In [57]:
driver = webdriver.Chrome(options)
driver.get("https://store.steampowered.com/app/730")

soup = BeautifulSoup(driver.page_source, 'html.parser')

reviews_container = soup.find('div', attrs={"data-featuretarget": "appreviews"})

if reviews_container:
    all_inner_content = reviews_container.prettify()

    print("Вся внутренность контейнера:")
    print("=" * 50)
    print(all_inner_content)
    with open('reviews_content.html', 'w', encoding='utf-8') as f:
        f.write(all_inner_content)
else:
    print("Контейнер с data-featuretarget='appreviews' не найден")

Вся внутренность контейнера:
<div data-featuretarget="appreviews" data-props='{"appid":730,"app_release_date":"1345568400","appname":"Counter-Strike 2","summary_num_positive_reviews":2172487,"summary_num_reviews":2515247}'>
</div>



In [2]:
!pip install httpx

In [9]:
from dataclasses import dataclass, field
from typing import List, Optional
from datetime import datetime

@dataclass
class SteamReviewsSummary:
    total_reviews: int = 0
    rating_value: int = 0
    best_rating: int = 0
    worst_rating: int = 0

@dataclass(frozen=True)
class SteamGame:
    title: str = ""
    release_date: str = ""
    developers: list[str] = field(default_factory=list)
    additional_info: dict[str] = field(default_factory=dict)
    description: str = ""

    price: Optional[float] = None
    currency: str = "USD"

    tags: List[str] = field(default_factory=list)
    reviews: SteamReviewsSummary = field(default_factory=SteamReviewsSummary)

In [43]:
import asyncio
from concurrent.futures import ThreadPoolExecutor

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import re, httpx

class SteamParser:
    def __init__(self, log_file='steam_parser.log', log_level=logging.INFO):
        self.driver = webdriver.Chrome(options)

        self.options = Options()

        prefs = {"profile.managed_default_content_settings.images": 2}
        self.options.add_experimental_option("prefs", prefs)

        self.options.page_load_strategy = 'eager'

        self.options.add_argument('--no-sandbox')
        self.options.add_argument('--disable-dev-shm-usage')
        self.options.add_argument("--headless")
        self.options.add_argument("--disable-gpu")

        self.logger = logging.getLogger('steam_parser')

    def parse_games_links(self, page: int) -> list[str]:
        self.logger.info(f"processing parsing game links from https://store.steampowered.com/search/?ignore_preferences=1&page={page}")

        self.driver.get(f"https://store.steampowered.com/search/?ignore_preferences=1&page={page}")

        html = self.driver.page_source

        soup = BeautifulSoup(html, 'html.parser')

        allLinks = []

        items = soup.find_all('div', id='search_resultsRows')
        for item in items:
            links = item.find_all('a')
            for link in links:
                url = link.get('href')
                if url:
                    allLinks.append(url.split('?')[0].rstrip('/'))

        return allLinks

    async def get_content_via_httpx(self, link: str, client: httpx.AsyncClient) -> list[BeautifulSoup, bool]:
        try:
                response = await client.get(link, timeout=10.0)

                if response.status_code != 200:
                    self.logger.error(f"Failed to fetch link {link}: Status {response.status_code}")
                    return None, False

                html = response.text
                soup = BeautifulSoup(html, 'html.parser')

                if not soup.find('div', id='appHubAppName') and soup.find('div', class_='age_gate'):
                    self.logger.info(f"Age check detected via requests for {link}")
                    return None, True

                return soup, False
        except Exception as e:
            self.logger.error(f"Requests error: {e}")
        return None, False

    async def parse_game_links_v2(self, page: int, client: httpx.AsyncClient) -> list[str]:
        url = f"https://store.steampowered.com/search/?ignore_preferences=1&page={page}"

        self.logger.info(f"Processing parsing game links from {url}")

        try:
                response = await client.get(url, timeout=10.0)

                if response.status_code != 200:
                    self.logger.error(f"Failed to fetch page {page}: Status {response.status_code}")
                    return []

                html = response.text
                soup = BeautifulSoup(html, 'html.parser')
                all_links = []

                search_results = soup.find('div', id='search_resultsRows')
                if not search_results:
                    self.logger.warning(f"No search results found on page {page}")
                    return []

                links = search_results.find_all('a')
                for link in links:
                    url = link.get('href')
                    if url:
                        url = url.split('?')[0].rstrip('/')
                        all_links.append(url)

                return all_links

        except Exception as e:
            self.logger.error(f"async parsing error on page {page}: {e}")
            return []

    def get_soup_via_selenium(self, link: str):
        self.logger.info(f"using Selenium for {link}")
        self.driver.get(link)

        try:
            age_select = WebDriverWait(self.driver, 3).until(
                EC.presence_of_element_located((By.ID, "ageYear"))
            )

            from selenium.webdriver.support.ui import Select
            Select(age_select).select_by_value("2000")

            view_page_btn = self.driver.find_element(By.ID, "view_product_page_btn")
            view_page_btn.click()

            WebDriverWait(self.driver, 5).until(
                EC.presence_of_element_located((By.ID, "appHubAppName"))
            )
        except:
            pass

        return BeautifulSoup(self.driver.page_source, 'html.parser')


    def parse_game_(self, link: str) -> SteamGame:
        self.driver.get_link(link)

        text = self.driver.page_source
        soup = BeautifulSoup(text, 'html.parser')

        try:
            price, currency = self.parse_game_price(soup, link)

            parsed_game = SteamGame(
                title=self.parse_name_game(soup, link),
                release_date=self.parse_release_date_game(soup),
                tags=self.parse_tags(soup),
                price=price,
                currency=currency,
                description=self.parse_game_description(soup),
                developers=self.parse_developers(soup),
                reviews=self.parse_reviews_summary(soup),
                additional_info=self.parse_additional_description(soup),
            )
            return parsed_game
        except Exception as e:
            self.logger.info(f"exception game processing {e}, location {e.__traceback__}")

        return None


    async def parse_game_v2(self, link: str, client: httpx.AsyncClient) -> SteamGame:
        result_httpx, neededToSelenium = await self.get_content_via_httpx(link)
        if result_httpx is None:
            self.logger.warning(f"game on {link} failed to fetch info, trying to selenium is {neededToSelenium}")

        soup = self.get_soup_via_selenium(link)

        try:
            price, currency = self.parse_game_price(soup, link)

            parsed_game = SteamGame(
                title=self.parse_name_game(soup, link),
                release_date=self.parse_release_date_game(soup),
                tags=self.parse_tags(soup),
                price=price,
                currency=currency,
                description=self.parse_game_description(soup),
                developers=self.parse_developers(soup),
                reviews=self.parse_reviews_summary(soup),
                additional_info=self.parse_additional_description(soup),
            )
            return parsed_game
        except Exception as e:
            self.logger.info(f"exception game processing {e}, location {e.__traceback__}")

        return None

    def parse_tags(self, soup):
        result = []
        tags = soup.find('div', class_='glance_tags popular_tags')
        for tag in tags.find_all('a'):
            result.append(tag.text.strip())
        return result

    def float_chars(self, s: str):
      return str.isdigit(s) or s == ","


    def parse_currency(self, text: str) -> str:
      currency_map = {
          '₽': 'RUB',
          'руб.': 'RUB',
          'rub': 'RUB',
          'р.': 'RUB',
          'nt$': 'TWD',
          '$': 'USD',
          'usd': 'USD',
          '€': 'EUR',
          'eur': 'EUR',
          '£': 'GBP',
          'gbp': 'GBP',
          '¥': 'JPY',
          'jpy': 'JPY',
          '₩': 'KRW',
          'krw': 'KRW',
          'rmb': 'CNY',
          '¥': 'CNY'
      }
      currency = "USD"
      for symbol, curr in currency_map.items():
          if symbol in text:
              currency = curr
              break
      return currency


    def parse_game_price(self, soup, link: str) -> list[float, str]:
        try:
            price_elem = soup.find('div', class_='game_purchase_price price').text.strip()
            if "free" in price_elem.lower():
              return 0, "USD"

            currency = self.parse_currency(price_elem.lower())

            cleaned_price = re.sub(r'[^\d,.]', '', price_elem)

            if ',' in cleaned_price and '.' in cleaned_price:
              cleaned_price = cleaned_price.replace(',', '')
            elif ',' in cleaned_price:
              if len(cleaned_price.split(',')[-1]) == 2:
                cleaned_price = cleaned_price.replace(',', '.')
              else:
                cleaned_price = cleaned_price.replace(',', '')

            if cleaned_price == "":
              return 0, "USD"

            return float(cleaned_price), currency
        except Exception as e:
            self.logger.info(f"game {link}, has no price exception {e}")

            return 0, "USD"

    def parse_additional_description(self, soup) -> list[str]:
        try:
            content_descriptors = soup.find('div', id='game_area_content_descriptors')
            return content_descriptors.text.strip()
        except Exception as e:
            self.logger.error(f"exception due {e}")
            return ""


    def parse_name_game(self, soup: BeautifulSoup, link) -> str:
        try:
            name = soup.find('div', id='appHubAppName')
            return name.text.strip()
        except AttributeError:
            self.logger.info(f"game {link} has wrong name tag")
            return ""

    def parse_release_date_game(self, soup: BeautifulSoup) -> str:
        release_date_elem = soup.find('div', class_='release_date')
        return release_date_elem.find('div', class_='date').text.strip()

    def parse_games_on_links(self, links: list[str]) -> list[SteamGame]:
        res = []
        for i in links:
            res_parse = self.parse_game(i)
            if res_parse is not None:
                res.append(res_parse)
        return res

    def parse_game_description(self, soup: BeautifulSoup) -> str:
        try:
            description_elem = soup.find('div', class_='game_description_snippet')
            return description_elem.text.strip()
        except Exception as e:
            self.logger.error(f"exception due {e}")
            return ""

    def parse_reviews_summary(self, soup: BeautifulSoup) -> SteamReviewsSummary:
        user_reviews_elem = soup.find('div', class_='user_reviews')
        reviews_aggr = user_reviews_elem.find('a',attrs={'itemprop': "aggregateRating"})

        def get_meta(prop):
            tag = reviews_aggr.find('meta', itemprop=prop)
            return tag.get('content', 0) if tag else 0

        return SteamReviewsSummary(
            total_reviews=int(get_meta('reviewCount')),
            rating_value=int(get_meta('ratingValue')),
            best_rating=int(get_meta('bestRating')),
            worst_rating=int(get_meta('worstRating')),
            )


    def parse_developers(self, soup: BeautifulSoup) -> list[str]:
        description_elem = soup.find('div', id="developers_list")

        result = []
        for dev in description_elem.find_all('a'):
            result.append(dev.text.strip())
        return result

In [83]:
SteamParser().parse_game("https://store.steampowered.com/app/730")

SteamGame(title='Counter-Strike 2', release_date='Aug 21, 2012', developers=['Valve'], additional_info='Mature Content Description\nThe developers describe the content like this:\n\n\t\t\tIncludes intense violence and blood.', description='For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2.', price=0, currency='USD', tags=['FPS', 'Shooter', 'Multiplayer', 'Competitive', 'Action', 'Team-Based', 'eSports', 'Tactical', 'First-Person', 'PvP', 'Online Co-Op', 'Co-op', 'Strategy', 'Military', 'War', 'Difficult', 'Trading', 'Realistic', 'Fast-Paced', 'Moddable'], reviews=SteamReviewsSummary(total_reviews=2515286, rating_value=9, best_rating=10, worst_rating=1))

In [11]:
SteamParser().parse_game("https://store.steampowered.com/app/570")

<coroutine object SteamParser.parse_game at 0x7b75568c8ac0>

In [45]:
parser = SteamParser()

for i in range(5):
    links = parser.parse_games_links(i+1)
links

['https://store.steampowered.com/app/2537590/Microsoft_Flight_Simulator_2024',
 'https://store.steampowered.com/app/1623730/Palworld',
 'https://store.steampowered.com/app/1286830/STAR_WARS_The_Old_Republic',
 'https://store.steampowered.com/app/4224910/Ready_or_Not_Boiling_Point',
 'https://store.steampowered.com/app/949230/Cities_Skylines_II',
 'https://store.steampowered.com/app/2129530/REANIMAL',
 'https://store.steampowered.com/app/3949040/RV_There_Yet',
 'https://store.steampowered.com/app/394360/Hearts_of_Iron_IV',
 'https://store.steampowered.com/app/632360/Risk_of_Rain_2',
 'https://store.steampowered.com/app/489830/The_Elder_Scrolls_V_Skyrim_Special_Edition',
 'https://store.steampowered.com/app/2592160/Dispatch',
 'https://store.steampowered.com/app/1384160/GUILTY_GEAR_STRIVE',
 'https://store.steampowered.com/app/2651280/Marvels_SpiderMan_2',
 'https://store.steampowered.com/app/250900/The_Binding_of_Isaac_Rebirth',
 'https://store.steampowered.com/app/1913370/OPERATOR',
 '

In [48]:
async def worker_links(parse_links_func, semaphore, page_num, client):
    async with semaphore:
        res = await parse_links_func(page_num, client)
        print(f"processed page {page_num}")
        return res


global all_our_links

async def main():
    async with httpx.AsyncClient(timeout=10.0) as client:
        pages_count = 50
        limit = 20
        semaphore = asyncio.Semaphore(limit)

        tasks = [worker_links(SteamParser().parse_game_links_v2, semaphore, i, client) for i in range(pages_count)]


        all_our_links = await asyncio.gather(*tasks)

await main()

/tmp/ipykernel_516/629520259.py:-1: RuntimeWarning: coroutine 'worker_links' was never awaited


WebDriverException: Message: Service /root/.cache/selenium/chromedriver/linux64/147.0.7727.56/chromedriver unexpectedly exited. Status code was: -2


In [93]:
steam_games = SteamParser().parse_games_on_links(links)
steam_games

INFO     - game https://store.steampowered.com/app/3321460/Crimson_Desert, has no price exception 'NoneType' object has no attribute 'text'
game https://store.steampowered.com/app/3321460/Crimson_Desert, has no price exception 'NoneType' object has no attribute 'text'
INFO     - game https://store.steampowered.com/app/3321460/Crimson_Desert has wrong name tag
INFO     - exception game processing 'NoneType' object has no attribute 'find', location <traceback object at 0x7db29954ae80>
INFO     - game https://store.steampowered.com/app/230410/Warframe, has no price exception 'NoneType' object has no attribute 'text'
game https://store.steampowered.com/app/230410/Warframe, has no price exception 'NoneType' object has no attribute 'text'
INFO     - game https://store.steampowered.com/app/230410/Warframe has wrong name tag
INFO     - exception game processing 'NoneType' object has no attribute 'find', location <traceback object at 0x7db290558fc0>
ERROR    - exception due 'NoneType' object ha

[SteamGame(title='Counter-Strike 2', release_date='Aug 21, 2012', developers=['Valve'], additional_info='Mature Content Description\nThe developers describe the content like this:\n\n\t\t\tIncludes intense violence and blood.', description='For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2.', price=0, currency='USD', tags=['FPS', 'Shooter', 'Multiplayer', 'Competitive', 'Action', 'Team-Based', 'eSports', 'Tactical', 'First-Person', 'PvP', 'Online Co-Op', 'Co-op', 'Strategy', 'Military', 'War', 'Difficult', 'Trading', 'Realistic', 'Fast-Paced', 'Moddable'], reviews=SteamReviewsSummary(total_reviews=2515286, rating_value=9, best_rating=10, worst_rating=1)),
 SteamGame(title='Slay the Spire 2', release_date='Mar 5, 2026', developers=['Mega Crit'], additional_info='', description='The iconic roguelike deckbuilder retur

In [28]:
st = SteamParser()
await st.parse_game("https://store.steampowered.com/app/3321460")

<!DOCTYPE html>
<html class=" responsive DesktopUI" lang="en"  >
<head>
	<meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
			<meta name="viewport" content="width=device-width,initial-scale=1">
		<meta name="theme-color" content="#171a21">
		<title>Crimson Desert on Steam</title>
	<link rel="shortcut icon" href="/favicon.ico" type="image/x-icon">

	
	
	<link href="https://store.fastly.steamstatic.com/public/shared/css/motiva_sans.css?v=YzJgj1FjzW34&amp;l=english&amp;_cdn=fastly" rel="stylesheet" type="text/css">
<link href="https://store.fastly.steamstatic.com/public/shared/css/shared_global.css?v=qbbUOQL2SbhT&amp;l=english&amp;_cdn=fastly" rel="stylesheet" type="text/css">
<link href="https://store.fastly.steamstatic.com/public/shared/css/buttons.css?v=BZhNEtESfYSJ&amp;l=english&amp;_cdn=fastly" rel="stylesheet" type="text/css">
<link href="https://store.fastly.steamstatic.com/public/css/v6/store.css?v=13CjnPuGjLMj&amp;l=english&amp;_cdn=fastly" rel="stylesheet" type

SteamGame(title='Crimson Desert', release_date='Mar 19, 2026', developers=['Pearl Abyss'], additional_info='AI Generated Content Disclosure\nThe developers describe how their game uses AI Generated Content like this:\n\n\t\t\t\t\tGenerative AI technology is used in a supplementary capacity during the creation of some 2D prop assets.\nAny such assets are replaced through our production pipeline by our art and development teams, ensuring they meet our quality standards and creative direction.', description='Crimson Desert is an open-world action-adventure set on the continent of Pywel. Join Kliff on his journey to rebuild the Greymane faction and to save the land from a looming threat. From vast wilderness and cities to ruins and the mysterious Abyss, forge your path through battles and discovery.', price=69.99, currency='USD', tags=['Open World', 'Action', 'Singleplayer', 'Adventure', 'Exploration', 'Combat', 'Dragons', 'Action-Adventure', 'Medieval', 'Third Person', 'Fantasy', 'RPG', '

In [30]:
!pip install pandas

In [84]:
import pandas as pd

df = pd.DataFrame([vars(u) for u in steam_games if u])

df

,title,release_date,developers,additional_info,description,price,currency,tags,reviews
0,Counter-Strike 2,"Aug 21, 2012",[Valve],Mature Content Description\nThe developers des...,"For over two decades, Counter-Strike has offer...",0.00,USD,"[FPS, Shooter, Multiplayer, Competitive, Actio...","SteamReviewsSummary(total_reviews=2515286, rat..."
1,Slay the Spire 2,"Mar 5, 2026",[Mega Crit],,The iconic roguelike deckbuilder returns. Craf...,24.99,USD,"[Strategy, Roguelike, Card Game, Deckbuilding,...","SteamReviewsSummary(total_reviews=41407, ratin..."
2,Apex Legends™,"Nov 5, 2020",[Respawn],,"Apex Legends is the award-winning, free-to-pla...",0.00,USD,"[Free to Play, Battle Royale, Multiplayer, FPS...","SteamReviewsSummary(total_reviews=439111, rati..."
3,ARC Raiders,"Oct 30, 2025",[Embark Studios],AI Generated Content Disclosure\nThe developer...,ARC Raiders is a multiplayer extraction advent...,39.99,USD,"[Extraction Shooter, Multiplayer, PvP, PvE, Th...","SteamReviewsSummary(total_reviews=183730, rati..."
4,Resident Evil 4,"Mar 23, 2023","[CAPCOM Co., Ltd.]",Mature Content Description\nThe developers des...,Survival is just the beginning. Six years have...,100.24,USD,"[Horror, Action, Survival Horror, Third-Person...","SteamReviewsSummary(total_reviews=62177, ratin..."
5,Marvel Rivals,"Dec 5, 2024",[NetEase Games],,Marvel Rivals is a Super Hero Team-Based PVP S...,0.00,USD,"[Free to Play, Multiplayer, Hero Shooter, Thir...","SteamReviewsSummary(total_reviews=274725, rati..."
6,Gray Zone Warfare,"Apr 30, 2024","[MADFINGER Games, a.s.]",Mature Content Description\nThe developers des...,Enter a high-stakes PvE-first open-world tacti...,0.00,USD,"[FPS, Tactical, Realistic, Extraction Shooter,...","SteamReviewsSummary(total_reviews=42509, ratin..."
7,War Thunder,"Aug 15, 2013",[Gaijin Entertainment],,War Thunder is the most comprehensive free-to-...,0.00,USD,"[Free to Play, Simulation, Vehicular Combat, C...","SteamReviewsSummary(total_reviews=305087, rati..."
8,Rust,"Feb 8, 2018",[Facepunch Studios],Mature Content Description\nThe developers des...,The only aim in Rust is to survive. Everything...,39.99,USD,"[Survival, Crafting, Multiplayer, Open World, ...","SteamReviewsSummary(total_reviews=517803, rati..."
9,Marathon,"Mar 5, 2026",[Bungie],,Scavenge the lost colony of Tau Ceti IV as a b...,39.99,USD,"[Extraction Shooter, PvP, Multiplayer, Sci-fi,...","SteamReviewsSummary(total_reviews=26956, ratin..."


In [99]:
import csv
import os

def object_to_dict(obj):
    result = {}
    for key, value in obj.__dict__.items():
        if hasattr(value, "__dict__"):
            nested = object_to_dict(value)
            for n_key, n_value in nested.items():
                result[f"{key}_{n_key}"] = n_value
        else:
            result[key] = value
    return result

steam_games_parsed = [object_to_dict(u) for u in steam_games]
fields = steam_games_parsed[0].keys()

with open('./data.csv', 'w', newline='', encoding='utf-8') as f:
    print(os.curdir, fields)
    print(os.path.abspath("data.csv"))

    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(steam_games_parsed)

SteamGame(title='Counter-Strike 2', release_date='Aug 21, 2012', developers=['Valve'], additional_info='Mature Content Description\nThe developers describe the content like this:\n\n\t\t\tIncludes intense violence and blood.', description='For over two decades, Counter-Strike has offered an elite competitive experience, one shaped by millions of players from across the globe. And now the next chapter in the CS story is about to begin. This is Counter-Strike 2.', price=0, currency='USD', tags=['FPS', 'Shooter', 'Multiplayer', 'Competitive', 'Action', 'Team-Based', 'eSports', 'Tactical', 'First-Person', 'PvP', 'Online Co-Op', 'Co-op', 'Strategy', 'Military', 'War', 'Difficult', 'Trading', 'Realistic', 'Fast-Paced', 'Moddable'], reviews=SteamReviewsSummary(total_reviews=2515286, rating_value=9, best_rating=10, worst_rating=1))
SteamReviewsSummary(total_reviews=2515286, rating_value=9, best_rating=10, worst_rating=1)
SteamGame(title='Slay the Spire 2', release_date='Mar 5, 2026', developer